**Note: When splitting a county into two model areas use following convention:**
- CO_FIPS = 9057 (9=added table + 0=SubareaID + 57=CO_FIPS)
- CO_FIPS = 9157 (9=added table + 1=SubareaID + 57=CO_FIPS)

In [1]:
import pandas as pd
import re
import numpy as np

In [2]:
category_dict = {
  # these categories use splits as defined in spreadsheet
  'POP': ['TOTPOP','GQPOP','HHPOP','POP0017','POP1864','POP65P'],
  'HH': ['HH'],
  'RETL': ['RETL','FOOD'],
  'IND': ['MANU','WSLE'],
  'OFFI': ['OFFI']#,
  # other categories are split programatically in this notebook
  #'FACTOR': ['GVED','HLTH','OTHR','AGRI','MING','CONST']
}

cat_expanded_df = (
    pd.DataFrame([
        {'CATEGORY': cat, 'VARIABLE': v}
        for cat, vars in category_dict.items()
        for v in vars
    ])
)

cat_expanded_df

,CATEGORY,VARIABLE
0,POP,TOTPOP
1,POP,GQPOP
2,POP,HHPOP
3,POP,POP0017
4,POP,POP1864
5,POP,POP65P
6,HH,HH
7,RETL,RETL
8,RETL,FOOD
9,IND,MANU


In [3]:
subarea_df = pd.DataFrame([
  ['WF'  ,1], # WF TDM
  ['UDOT',0]  # USTM
], columns=("MODEL","SUBAREAID"))
subarea_df

,MODEL,SUBAREAID
0,WF,1
1,UDOT,0


In [4]:
from _counties import get_counties_df
counties_df = get_counties_df()
counties_df

,CO_FIPS,CO_NAME
0,5,CACHE
1,13,DUCHESNE
2,11,DAVIS
3,27,MILLARD
4,51,WASATCH
5,3,BOX ELDER
6,57,WEBER
7,23,JUAB
8,39,SANPETE
9,53,WASHINGTON


# Split Box Elder and Weber counties

In [5]:
df_box = pd.read_csv(
    "data/Splits-Data-From-Josh-20251121/REMM-WF-Model-Area-Splits (RTP27) - Box Elder.csv"
)
df_box.rename(columns={'TOTALEMP_PCT_UDOT':'TOTAL_PCT_UDOT'}, inplace=True)
df_box['TOTAL_PCT_WF'] = np.round(1 - df_box['TOTAL_PCT_UDOT'],3)
df_box["CO_NAME"] = "BOX ELDER"

df_weber = pd.read_csv(
    "data/Splits-Data-From-Josh-20251121/REMM-WF-Model-Area-Splits (RTP27) - Weber.csv"
)
df_weber["CO_NAME"] = "WEBER"

wf_model_area_splits_df = pd.concat([df_box, df_weber], ignore_index=True)

wf_model_area_splits_df.rename(columns={'Year':'YEAR'}, inplace=True)

# add underscore before PCT where missing
wf_model_area_splits_df.columns = [
    re.sub(r'(?<!_)PCT', r'_PCT', col)
    for col in wf_model_area_splits_df.columns
]
wf_model_area_splits_df

,YEAR,HH_PCT_WF,HH_PCT_UDOT,POP_PCT_WF,POP_PCT_UDOT,OFFI_PCT_WF,OFFI_PCT_UDOT,RETL_PCT_WF,RETL_PCT_UDOT,IND_PCT_WF,IND_PCT_UDOT,TOTAL_PCT_UDOT,TOTAL_PCT_WF,CO_NAME
0,2023,0.541,0.459,0.530,0.470,0.714,0.286,0.661,0.339,0.378,0.622,0.463,0.537,BOX ELDER
1,2024,0.541,0.459,0.529,0.471,0.713,0.287,0.665,0.335,0.381,0.619,0.461,0.539,BOX ELDER
2,2025,0.540,0.460,0.528,0.472,0.713,0.287,0.668,0.332,0.383,0.617,0.459,0.541,BOX ELDER
3,2026,0.539,0.461,0.528,0.472,0.713,0.287,0.671,0.329,0.386,0.614,0.458,0.542,BOX ELDER
4,2027,0.539,0.461,0.527,0.473,0.713,0.287,0.673,0.327,0.388,0.612,0.456,0.544,BOX ELDER
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81,2061,0.961,0.039,0.967,0.033,0.962,0.038,0.999,0.001,0.994,0.006,0.022,0.978,WEBER
82,2062,0.961,0.039,0.967,0.033,0.962,0.038,0.999,0.001,0.994,0.006,0.022,0.978,WEBER
83,2063,0.961,0.039,0.967,0.033,0.962,0.038,0.999,0.001,0.994,0.006,0.022,0.978,WEBER
84,2064,0.961,0.039,0.967,0.033,0.962,0.038,0.999,0.001,0.994,0.006,0.022,0.978,WEBER


In [6]:
# Check for Negatives
# select only numeric columns
num_df = wf_model_area_splits_df.select_dtypes(include='number')

# check for any negative value in numeric fields
if (num_df < 0).any().any():
  print("NEGATIVES!")
else:
  print("No Negatives! Good to go!")


No Negatives! Good to go!


In [7]:
wf_model_area_splits_long_df = wf_model_area_splits_df.melt(
    id_vars=["CO_NAME","YEAR"],
    var_name="COLNAME",
    value_name="PERCENT"
)

parts = wf_model_area_splits_long_df["COLNAME"].str.split("_", expand=True)

wf_model_area_splits_long_df["CATEGORY"] = parts[0]
wf_model_area_splits_long_df["MODEL"] = parts[2]

wf_model_area_splits_long_df.drop(columns=['COLNAME'], inplace=True)

wf_model_area_splits_long_df = wf_model_area_splits_long_df.merge(
    cat_expanded_df,
    on="CATEGORY",
    how="left"
)

# add subarea id
wf_model_area_splits_long_df = wf_model_area_splits_long_df.merge(subarea_df,on='MODEL')

# add CO_FIPS
wf_model_area_splits_long_df = wf_model_area_splits_long_df.merge(counties_df,on='CO_NAME')

# CO_FIPS_split
wf_model_area_splits_long_df['CO_FIPS_split'] = 9000 + wf_model_area_splits_long_df['SUBAREAID'] * 100 + wf_model_area_splits_long_df['CO_FIPS']

wf_model_area_splits_long_df

,CO_NAME,YEAR,PERCENT,CATEGORY,MODEL,VARIABLE,SUBAREAID,CO_FIPS,CO_FIPS_split
0,BOX ELDER,2023,0.541,HH,WF,HH,1,3,9103
1,BOX ELDER,2024,0.541,HH,WF,HH,1,3,9103
2,BOX ELDER,2025,0.540,HH,WF,HH,1,3,9103
3,BOX ELDER,2026,0.539,HH,WF,HH,1,3,9103
4,BOX ELDER,2027,0.539,HH,WF,HH,1,3,9103
...,...,...,...,...,...,...,...,...,...
2231,WEBER,2061,0.978,TOTAL,WF,NaN,1,57,9157
2232,WEBER,2062,0.978,TOTAL,WF,NaN,1,57,9157
2233,WEBER,2063,0.978,TOTAL,WF,NaN,1,57,9157
2234,WEBER,2064,0.978,TOTAL,WF,NaN,1,57,9157


In [8]:
# Get TOTAL

wf_model_area_splits_total_df = wf_model_area_splits_long_df.drop(columns=['VARIABLE']).copy()
wf_model_area_splits_total_df = wf_model_area_splits_total_df[wf_model_area_splits_total_df['CATEGORY']=="TOTAL"]
wf_model_area_splits_total_df

,CO_NAME,YEAR,PERCENT,CATEGORY,MODEL,SUBAREAID,CO_FIPS,CO_FIPS_split
2064,BOX ELDER,2023,0.463,TOTAL,UDOT,0,3,9003
2065,BOX ELDER,2024,0.461,TOTAL,UDOT,0,3,9003
2066,BOX ELDER,2025,0.459,TOTAL,UDOT,0,3,9003
2067,BOX ELDER,2026,0.458,TOTAL,UDOT,0,3,9003
2068,BOX ELDER,2027,0.456,TOTAL,UDOT,0,3,9003
...,...,...,...,...,...,...,...,...
2231,WEBER,2061,0.978,TOTAL,WF,1,57,9157
2232,WEBER,2062,0.978,TOTAL,WF,1,57,9157
2233,WEBER,2063,0.978,TOTAL,WF,1,57,9157
2234,WEBER,2064,0.978,TOTAL,WF,1,57,9157


# Population

In [9]:
pop_df = pd.concat([pd.read_csv('intermediate/tdm_population_by_county_2023_2024.csv'),pd.read_csv('intermediate/tdm_population_by_county_2025+.csv')]).rename(columns={'TDM_VARIABLE':'VARIABLE'})
pop_df

,CO_FIPS,YEAR,VARIABLE,VALUE,CO_NAME
0,1,2023,TOTPOP,7314,BEAVER
1,3,2023,TOTPOP,61251,BOX ELDER
2,5,2023,TOTPOP,141700,CACHE
3,7,2023,TOTPOP,20654,CARBON
4,9,2023,TOTPOP,998,DAGGETT
...,...,...,...,...,...
8318,57,2061,POP65P,88319,WEBER
8319,57,2062,POP65P,90098,WEBER
8320,57,2063,POP65P,91765,WEBER
8321,57,2064,POP65P,93217,WEBER


In [10]:
import numpy as np
import pandas as pd

# 1. Base data: keep as-is for original records
orig_df = pop_df[['CO_FIPS', 'VARIABLE', 'YEAR', 'VALUE']].copy()
orig_df['CO_FIPS_split'] = orig_df['CO_FIPS']

# 2. Merge in split info (but we'll only use this to build the split rows)
split_df = pop_df.merge(
    wf_model_area_splits_long_df,
    on=['CO_FIPS', 'VARIABLE', 'YEAR'],
    how='inner',        # keep all original rows
    suffixes=('', '_split')
)

# Scale VALUE by PERCENT
split_df['VALUE'] = split_df['VALUE'] * split_df['PERCENT']

# Banker's rounding to int
split_df['VALUE'] = np.round(split_df['VALUE']).astype(int)

# Keep just the core columns in the same format as original
split_df = split_df[['CO_FIPS_split','YEAR','VARIABLE','VALUE','CO_FIPS']]

# 4. Combine original + split rows
pop_split_df = pd.concat([orig_df, split_df], ignore_index=True)

# Clean up types / sort if needed
pop_split_df['CO_FIPS_split'] = pop_split_df['CO_FIPS_split'].astype(int)

pop_split_df = pop_split_df[['CO_FIPS_split','YEAR','VARIABLE','VALUE','CO_FIPS']]

pop_split_df = pop_split_df.sort_values(['CO_FIPS_split', 'YEAR','VARIABLE']).reset_index(drop=True)
pop_split_df

,CO_FIPS_split,YEAR,VARIABLE,VALUE,CO_FIPS
0,1,2023,GQPOP,300,1
1,1,2023,HH,2381,1
2,1,2023,HHPOP,7014,1
3,1,2023,POP0017,2019,1
4,1,2023,POP1864,4185,1
...,...,...,...,...,...
9928,9157,2065,HHPOP,367411,57
9929,9157,2065,POP0017,59084,57
9930,9157,2065,POP1864,222289,57
9931,9157,2065,POP65P,91658,57


# Employment

In [11]:
# EASY:
#
# RETL = (County RETL) x (RETL_PCT)
# FOOD = (County FOOD) x (RETL_PCT)
# MANU = (County MANU) x (IND_PCT)
# WSLE = (County WSLE) x (IND_PCT)
# OFFI = (County OFFI) x (OFFI_PCT)

In [12]:
emp_df = pd.read_csv('intermediate/tdm_employment_by_county.csv').rename(columns={'TDM_SECTOR':'VARIABLE'}).drop(columns=['CO_NAME'])
emp_df

,CO_FIPS,YEAR,VARIABLE,EMPCOUNT
0,1,2023,AGRI,738
1,1,2023,CONS,278
2,1,2023,FOOD,517
3,1,2023,GVED,810
4,1,2023,HLTH,160
...,...,...,...,...
13712,57,2065,MING,169
13713,57,2065,OFFI,14763
13714,57,2065,OTHR,56307
13715,57,2065,RETL,20376


In [13]:
# SPLIT EASY PERCENT

# Merge in the split definitions
emp_split_easy_df = emp_df.merge(
    wf_model_area_splits_long_df,
    on=['CO_FIPS', 'VARIABLE', 'YEAR'],
    how='inner',              # keep all original rows
    suffixes=('', '_split')
)

# Scale EMPCOUNT by PERCENT
#    (Assumes PERCENT is 0–1; if 0–100, divide by 100)
emp_split_easy_df['EMPCOUNT']    = np.round(emp_split_easy_df['EMPCOUNT']    * emp_split_easy_df['PERCENT']).astype(int)

# Keep only the core columns, same structure as original
emp_split_easy_df = emp_split_easy_df[['CO_FIPS_split','CO_FIPS', 'VARIABLE', 'YEAR', 'EMPCOUNT']]

emp_split_easy_df = (
    emp_split_easy_df
    .sort_values(['CO_FIPS_split', 'YEAR', 'VARIABLE'])
    .reset_index(drop=True)
)

emp_split_easy_df

,CO_FIPS_split,CO_FIPS,VARIABLE,YEAR,EMPCOUNT
0,9003,3,FOOD,2023,678
1,9003,3,MANU,2023,4982
2,9003,3,OFFI,2023,390
3,9003,3,RETL,2023,1038
4,9003,3,WSLE,2023,1965
...,...,...,...,...,...
855,9157,57,FOOD,2065,9894
856,9157,57,MANU,2065,28335
857,9157,57,OFFI,2065,14202
858,9157,57,RETL,2065,20356


In [14]:
# HARD:
#
# For remaining sectors, find the factor to apply: 
# 
# Get the total number of jobs for each model area = [CO_TOTAL] x [TOTAL_PCT]
# The sum of all the jobs in each model area should equal the total number of jobs for each model area.
# MA_TOTAL = [CO_TOTAL] x [TOTAL_PCT]    =    Sum of all job sectors
# MA_RETL + MA_FOOD + MA_MANU + MA_WSLE + MA_OFFI + MA_GVED + MA_HLTH + MA_OTHR + MA_AGRI + MA_MING + MA_CONS
#
# Model Area Totals are Factor x County Totals
# MA_GVED + MA_HLTH + MA_OTHR + MA_AGRI + MA_MING + MA_CONS = FACTOR_HARD * (CO_GVED + CO_HLTH + CO_OTHR + CO_AGRI + CO_MING + CO_CONS)
# 
## The first five TDM job sectors have their own percentages. We need to find the FACTOR_HARD to apply for all other sectors.
# MA_TOTAL = (MA_RETL + MA_FOOD + MA_MANU + MA_WSLE + MA_OFFI) + (FACTOR_HARD  x (CO_GVED + CO_HLTH + CO_OTHR + CO_AGRI + CO_MING + CO_CONS))
# 
# FACTOR_HARD = (MA_TOTAL - (MA_RETL + MA_FOOD + MA_MANU + MA_WSLE + MA_OFFI)) / (CO_GVED + CO_HLTH + CO_OTHR + CO_AGRI + CO_MING + CO_CONS)
# 
# MA_GVED = CO_GVED x FACTOR_HARD
# MA_HLTH = CO_HLTH x FACTOR_HARD
# MA_OTHR = CO_OTHR x FACTOR_HARD
# MA_AGRI = CO_AGRI x FACTOR_HARD
# MA_MING = CO_MING x FACTOR_HARD
# MA_CONS = CO_CONS x FACTOR_HARD
#
# MA_EASY = MA_RETL + MA_FOOD + MA_MANU + MA_WSLE + MA_OFFI
# CO_HARD = CO_GVED + CO_HLTH + CO_OTHR + CO_AGRI + CO_MING + CO_CONS

# FACTOR_HARD = (MA_TOTAL - MA_EASY) / CO_HARD


In [15]:
# MA_TOTAL = [CO_TOTAL] x [TOTAL_PCT]

emp_split_MA_total_df = emp_df.groupby(['CO_FIPS','YEAR']).agg(EMPTOTAL=('EMPCOUNT','sum')).reset_index()
emp_split_MA_total_df = pd.merge(emp_split_MA_total_df,wf_model_area_splits_total_df, on=['CO_FIPS','YEAR'])
emp_split_MA_total_df['MA_TOTAL'] = np.round(emp_split_MA_total_df['EMPTOTAL'] * emp_split_MA_total_df['PERCENT'],0).astype(int)
emp_split_MA_total_df = emp_split_MA_total_df[['CO_FIPS_split','YEAR','MA_TOTAL','CO_FIPS']]
emp_split_MA_total_df = emp_split_MA_total_df.sort_values(['CO_FIPS_split','YEAR'])
emp_split_MA_total_df

,CO_FIPS_split,YEAR,MA_TOTAL,CO_FIPS
0,9003,2023,15913,3
2,9003,2024,15791,3
4,9003,2025,15962,3
6,9003,2026,16063,3
8,9003,2027,16149,3
...,...,...,...,...
163,9157,2061,214648,57
165,9157,2062,215666,57
167,9157,2063,216687,57
169,9157,2064,217713,57


In [16]:
# MA_EASY = MA_RETL + MA_FOOD + MA_MANU + MA_WSLE + MA_OFFI

_df = emp_split_easy_df.pivot(index=['CO_FIPS_split','CO_FIPS','YEAR'],values='EMPCOUNT',columns='VARIABLE').reset_index()
_df['MA_EASY'] = _df['RETL'] + _df['FOOD'] + _df['MANU'] + _df['WSLE'] + _df['OFFI']
emp_split_MA_easy_df = _df[['CO_FIPS_split','YEAR','MA_EASY']]
emp_split_MA_easy_df

VARIABLE,CO_FIPS_split,YEAR,MA_EASY
0,9003,2023,9053
1,9003,2024,8613
2,9003,2025,8709
3,9003,2026,8711
4,9003,2027,8744
...,...,...,...
167,9157,2061,82847
168,9157,2062,83173
169,9157,2063,83502
170,9157,2064,83832


In [17]:
# CO_HARD = CO_GVED + CO_HLTH + CO_OTHR + CO_AGRI + CO_MING + CO_CONS

_df = emp_df.pivot(index=['CO_FIPS','YEAR'], columns='VARIABLE', values='EMPCOUNT').reset_index()
_df['CO_HARD'] = _df['GVED'] + _df['HLTH'] + _df['OTHR'] + _df['AGRI'] + _df['MING'] + _df['CONS']
emp_CO_hard_df = _df[['CO_FIPS','YEAR','CO_HARD']]
emp_CO_hard_df

VARIABLE,CO_FIPS,YEAR,CO_HARD
0,1,2023,3025
1,1,2024,3216
2,1,2025,3357
3,1,2026,3524
4,1,2027,3546
...,...,...,...
1242,57,2061,135815
1243,57,2062,136526
1244,57,2063,137237
1245,57,2064,137951


In [18]:
# FACTOR_HARD = (MA_TOTAL - MA_EASY) / CO_HARD

_df = pd.merge(pd.merge(emp_split_MA_total_df,emp_split_MA_easy_df,on=['CO_FIPS_split','YEAR']),emp_CO_hard_df,on=['CO_FIPS','YEAR'])
_df['FACTOR_HARD'] = np.round((_df['MA_TOTAL'] - _df['MA_EASY']) / _df['CO_HARD'],4)
display(_df)
emp_split_hard_factors_df = _df[['CO_FIPS_split','CO_FIPS','YEAR','FACTOR_HARD']]
display(emp_split_hard_factors_df)

,CO_FIPS_split,YEAR,MA_TOTAL,CO_FIPS,MA_EASY,CO_HARD,FACTOR_HARD
0,9003,2023,15913,3,9053,16776,0.4089
1,9003,2024,15791,3,8613,17316,0.4145
2,9003,2025,15962,3,8709,17614,0.4118
3,9003,2026,16063,3,8711,17835,0.4122
4,9003,2027,16149,3,8744,18067,0.4099
...,...,...,...,...,...,...,...
167,9157,2061,214648,57,82847,135815,0.9704
168,9157,2062,215666,57,83173,136526,0.9705
169,9157,2063,216687,57,83502,137237,0.9705
170,9157,2064,217713,57,83832,137951,0.9705


,CO_FIPS_split,CO_FIPS,YEAR,FACTOR_HARD
0,9003,3,2023,0.4089
1,9003,3,2024,0.4145
2,9003,3,2025,0.4118
3,9003,3,2026,0.4122
4,9003,3,2027,0.4099
...,...,...,...,...
167,9157,57,2061,0.9704
168,9157,57,2062,0.9705
169,9157,57,2063,0.9705
170,9157,57,2064,0.9705


In [19]:
# check if all 1
_check_df = emp_split_hard_factors_df.groupby(['CO_FIPS','YEAR']).agg(SUM1=('FACTOR_HARD','sum')).reset_index()

# find rows that do NOT sum to 1
bad_rows = _check_df[_check_df['SUM1'] != 1]

if bad_rows.empty:
    print("EVERYTHING CHECKS OUT!")
else:
    print("WARNING: Some groups do not sum to 1:")
    display(bad_rows)

EVERYTHING CHECKS OUT!


In [20]:
# MA_GVED = CO_GVED x FACTOR_HARD
# MA_HLTH = CO_HLTH x FACTOR_HARD
# MA_OTHR = CO_OTHR x FACTOR_HARD
# MA_AGRI = CO_AGRI x FACTOR_HARD
# MA_MING = CO_MING x FACTOR_HARD
# MA_CONS = CO_CONS x FACTOR_HARD

_df = pd.merge(emp_split_hard_factors_df,emp_df.pivot(index=['CO_FIPS','YEAR'], columns='VARIABLE', values='EMPCOUNT').reset_index(), on=['CO_FIPS','YEAR'])
_df['MA_GVED'] = np.round(_df['GVED'] * _df['FACTOR_HARD'],0).astype(int)
_df['MA_HLTH'] = np.round(_df['HLTH'] * _df['FACTOR_HARD'],0).astype(int)
_df['MA_OTHR'] = np.round(_df['OTHR'] * _df['FACTOR_HARD'],0).astype(int)
_df['MA_AGRI'] = np.round(_df['AGRI'] * _df['FACTOR_HARD'],0).astype(int)
_df['MA_MING'] = np.round(_df['MING'] * _df['FACTOR_HARD'],0).astype(int)
_df['MA_CONS'] = np.round(_df['CONS'] * _df['FACTOR_HARD'],0).astype(int)
display(_df)

emp_split_hard_df = _df[['CO_FIPS_split','CO_FIPS','YEAR','MA_GVED','MA_HLTH','MA_OTHR','MA_AGRI','MA_MING','MA_CONS']].copy()
emp_split_hard_df.rename(columns={'MA_GVED':'GVED','MA_HLTH':'HLTH','MA_OTHR':'OTHR','MA_AGRI':'AGRI','MA_MING':'MING','MA_CONS':'CONS'}, inplace=True)
emp_split_hard_df = emp_split_hard_df.melt(id_vars=['CO_FIPS_split','CO_FIPS','YEAR'], var_name="VARIABLE", value_name="EMPCOUNT")
emp_split_hard_df

,CO_FIPS_split,CO_FIPS,YEAR,FACTOR_HARD,AGRI,CONS,FOOD,GVED,HLTH,MANU,...,OFFI,OTHR,RETL,WSLE,MA_GVED,MA_HLTH,MA_OTHR,MA_AGRI,MA_MING,MA_CONS
0,9003,3,2023,0.4089,1694,2747,2000,3784,2148,8010,...,1364,6315,3061,3159,1547,878,2582,693,36,1123
1,9003,3,2024,0.4145,1697,3238,1838,4063,2267,7404,...,1427,5953,3082,3186,1684,940,2468,703,41,1342
2,9003,3,2025,0.4118,1693,3303,1836,4129,2353,7496,...,1443,6038,3090,3297,1700,969,2486,697,40,1360
3,9003,3,2026,0.4122,1692,3371,1849,4158,2418,7517,...,1429,6095,3079,3364,1714,997,2512,697,42,1390
4,9003,3,2027,0.4099,1685,3445,1852,4199,2474,7538,...,1418,6160,3103,3436,1721,1014,2525,691,43,1412
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
167,9157,57,2061,0.9704,1364,17958,9927,34551,26598,27914,...,14439,55173,20023,11358,33528,25811,53540,1324,166,17426
168,9157,57,2062,0.9705,1357,18037,9922,34710,26807,28056,...,14524,55444,20108,11381,33686,26016,53808,1317,166,17505
169,9157,57,2063,0.9705,1350,18110,9917,34867,27015,28204,...,14605,55725,20194,11404,33838,26218,54081,1310,165,17576
170,9157,57,2064,0.9705,1344,18178,9910,35024,27221,28353,...,14685,56014,20284,11427,33991,26418,54362,1304,165,17642


,CO_FIPS_split,CO_FIPS,YEAR,VARIABLE,EMPCOUNT
0,9003,3,2023,GVED,1547
1,9003,3,2024,GVED,1684
2,9003,3,2025,GVED,1700
3,9003,3,2026,GVED,1714
4,9003,3,2027,GVED,1721
...,...,...,...,...,...
1027,9157,57,2061,CONS,17426
1028,9157,57,2062,CONS,17505
1029,9157,57,2063,CONS,17576
1030,9157,57,2064,CONS,17642


In [21]:
# add split id to original employment
emp_orig_df = emp_df.copy()
emp_orig_df['CO_FIPS_split'] = emp_orig_df['CO_FIPS']

# combine all dfs
emp_split_combined_df = pd.concat([emp_split_easy_df, emp_split_hard_df, emp_orig_df]).rename(columns={'EMPCOUNT':'VALUE'})
emp_split_combined_df = emp_split_combined_df[['CO_FIPS_split','YEAR','VARIABLE','VALUE','CO_FIPS']]
emp_split_combined_df = emp_split_combined_df.sort_values(['CO_FIPS_split','YEAR','VARIABLE'])
emp_split_combined_df

,CO_FIPS_split,YEAR,VARIABLE,VALUE,CO_FIPS
0,1,2023,AGRI,738,1
1,1,2023,CONS,278,1
2,1,2023,FOOD,517,1
3,1,2023,GVED,810,1
4,1,2023,HLTH,160,1
...,...,...,...,...,...
859,9157,2065,MING,164,57
857,9157,2065,OFFI,14202,57
515,9157,2065,OTHR,54646,57
858,9157,2065,RETL,20356,57


In [22]:
pop_split_df.to_csv('intermediate/split_pop.csv', index=False)
emp_split_combined_df.to_csv('intermediate/split_emp.csv', index=False)